<a href="https://colab.research.google.com/github/jrgreen7/SYSC4906/blob/master/W2025/Tutorials/T8/Tutorial-8_Seq2Seq.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 8 - seq2seq - Subtraction

**Semester:** Winter 2026

**Adapted by:** [Kevin Dick](https://kevindick.ai/), [Igor Bogdanov](igorbogdanov@cmail.carleton.ca)

**Adapted from:** [seq2seq Tutorial](https://github.com/lukas/ml-class/blob/master/videos/seq2seq/train.py) originally from this [Keras Blog](https://blog.keras.io/a-ten-minute-introduction-to-sequence-to-sequence-learning-in-keras.html).
---

## Extra: the Douglas R. Hofstadter `pq`-system as a `seq2seq` task

For those familiar with Dr. Douglas Hofstadter's masterpiece [**(GEB) Gödel, Escher, Bach: and Eternal Golden Braid**](https://www.physixfan.com/wp-content/files/GEBen.pdf), the `pq`-system that Hofstader leverages heavily throughout the book can also be leveraged as an example of a `seq2seq` task.

More formally, the `pq`-system has only three disting symbols: `p`, `q`, and `-` and these are used in combination to generate statements/theorems for this system such as:

`--p---q-----`

`-p-q--`

`----------p-q-----------`

When you look at these example strings, can you *discern a meaning* for what the symbols `p`, `q`, and `-` stand for? As a human, we might try to identify a pattern within a large number of these statements and hope to deduce a pattern that allows us to generate new and valid statements within this system. 

### Excerpt from GEB (Chapter II: Isomorphisms Induce Meaning):

> Perhaps you have already thought to yourself that the `pq`-theorems are like additions. The string `--p---q-----` is a theorem because 2 plus 3 equals 5. It could even occur to you that the theorem `--p---q-----` is a statement, written in an odd notation, whose meaning is that **2 plus 3 is 5**. Is this a reasonable way to look at things? Well, I deliberately chose 'p' to remind you of 'plus',and 'q' to remind you of 'equals' . . . So, does the string `--p---q-----` actually mean "2 plus 3 equals 5"?

Aside: GEB is **strongly recommended** to those with deep interests at the intersection of *mathematics, artificial intelligence, philosophy, cognition, musical theory, and the arts.*

For the purposes of understanding the utility of `seq2seq` on solving arbitrary **string translation** tasks, this is precisely what a machine learning algorithm must do. 

**Presented with thousands of examples of valid query strings and their targets, the model learns an internal representation that allows it to correctly map the meaning of an assembly of sybmols into an alternative and valid representation.**

---

Similar to the examples above that generate example mathematical strings and have the model learn to "translate" that input string into its resulting output, we will generate pairs of strings valid in Hofstadter's `pq`-system:

**Example:** Input `x="---p--"` with target `y="q-----"`



## Encoding/Decoding Utility

Utility Class for encoding-decoding characters ('0123456789+ ') into one-hot matrices:

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────

from keras.models import Sequential                                # Linear stack of layers (our model type)
from keras.layers import LSTM, TimeDistributed, RepeatVector, Dense  # All layer types used in the seq2seq model
import numpy as np                                                 # Array math — used everywhere


# ── Character Lookup Table ────────────────────────────────────────────────────
# Identical to Parts 1(a) and 1(b) except this instance is initialized with a
# minimal 4-character alphabet: '-pq ' instead of digits and operators.
# '-' = a dash (represents a number's magnitude by count)
# 'p' = the pq-system's "plus" operator (separates the two operands)
# 'q' = the pq-system's "equals" operator (prefixes the answer)
# ' ' = padding

class CharacterTable(object):
    """Given a set of characters:
    + Encode them to a one hot integer representation
    + Decode the one hot integer representation to their character output
    + Decode a vector of probabilities to their character output
    """

    def __init__(self, chars):
        """Initialize character table.
        # Arguments
            chars: Characters that can appear in the input.
        """
        # Sort and deduplicate so the mapping is deterministic across runs
        self.chars = sorted(set(chars))

        # char → index:  e.g., {' ': 0, '-': 1, 'p': 2, 'q': 3}
        self.char_indices = dict((c, i) for i, c in enumerate(self.chars))

        # index → char:  the reverse lookup
        self.indices_char = dict((i, c) for i, c in enumerate(self.chars))

    def encode(self, C, num_rows):
        """One hot encode given string C.
        # Arguments
            num_rows: Number of rows in the returned one hot encoding. This is
                used to keep the # of rows for each data the same.
        """
        # Start with an all-zero matrix of shape (num_rows, vocab_size)
        # Each row will hold the one-hot vector for one character
        x = np.zeros((num_rows, len(self.chars)))

        # For each character in the string, flip its column to 1
        # e.g., 'p' → row i becomes [0, 0, 1, 0]
        for i, c in enumerate(C):
            x[i, self.char_indices[c]] = 1

        # Rows beyond len(C) stay all-zero (implicit padding)
        return x

    def decode(self, x, calc_argmax=True):
        if calc_argmax:
            # Each row is either a one-hot vector or a softmax probability
            # distribution — argmax picks the index of the highest value,
            # i.e., the most likely character at each timestep
            x = x.argmax(axis=-1)

        # Debug prints left in from development — safe to remove in production
        # print("hi")
        # print(x)
        # print(x.shape)
        print(len(self.indices_char))   # Prints vocab size (4) on every decode call

        # Map each index back to its character and join into a plain string
        # e.g., [2, 1, 1, 3, 1, 1, 1, 0...] → '--p--q----  ...'
        return ''.join(self.indices_char[x] for x in x)


## Defining Dataset Parameters

In [ ]:
# ── pq-System Parameters ──────────────────────────────────────────────────────

training_size = 1000   # Total number of pq-system examples
digits        = 45    # A "digit" here is a DASH, not a numeric digit.
                        # Allows sequences of up to 1000 dashes per operand —
                        # representing numbers 1 through 1000 in unary.
hidden_size   = 128     # Dimensionality of the RNN's internal state vector
batch_size    = 128     # Number of examples processed per gradient update

# maxlen must accommodate the longest possible string: 1000 dashes + 'p' + 1000 dashes
# = 2001 characters. This is vastly longer than Parts 1(a) and 1(b).
maxlen = digits + 1 + digits   # = 91

## Dataset Generation: Subtraction Problems

In [ ]:
# The entire vocabulary is just 4 symbols — the smallest alphabet of all three parts.
# '-' for unary magnitude, 'p' for the operator, 'q' for the answer prefix, ' ' for padding.
chars  = '-pq '
ctable = CharacterTable(chars)

# Empty containers — filled by the generation loop below
questions = []
expected  = []
seen      = set()    # Tracks (a, b) pairs already generated to avoid duplicates


# ── Data Generation ───────────────────────────────────────────────────────────

# Keep generating examples until we hit our target dataset size (50,000)
print('Generating data...')
while len(questions) < training_size:

    # Lambda that builds one random unary string:
    # Instead of a random integer, this generates a random-length string of dashes.
    # e.g., f() might return '--' (representing 2) or '-----' (representing 5).
    # Length is between 1 and digits (1 to 1000).
    f = lambda: ''.join('-' for i in range(np.random.randint(1, digits + 1)))

    # Generate two independent random unary strings
    a, b = f(), f()

    # Deduplicate: since '--p---' and '---p--' encode the same pair of magnitudes,
    # sort before hashing to avoid near-duplicate training examples.
    key = tuple(sorted((a, b)))
    if key in seen:
        continue
    seen.add(key)   # Mark this pair as used

    # Build the raw question string using pq-system notation.
    # e.g., a='--', b='---' → q = '--p---'
    # 'p' acts as the separator between the two unary operands.
    q = '{}p{}'.format(a, b)

    # Pad with trailing spaces so every question is exactly maxlen=2001 characters.
    query = q + ' ' * (maxlen - len(q))

    # The answer in pq-system notation: 'q' followed by (len(a) + len(b)) dashes.
    # e.g., '--p---' → 'q-----'  (2 + 3 = 5 dashes after 'q')
    # This is the entire "computation" — just concatenating the dash counts.
    ans = 'q' + '-' * (len(a) + len(b))

    # Pad the answer to maxlen as well.
    # Key difference from Parts 1(a) and 1(b): BOTH x and y have the same
    # length (maxlen=2001) because answers can be nearly as long as inputs.
    ans += ' ' * (maxlen - len(ans))

    # No REVERSE step — input is stored as-is
    questions.append(query)
    expected.append(ans)

print('Total addition questions:', len(questions))


## Converting QA Dataset to NumPy Array

In [ ]:
# ── Vectorization ─────────────────────────────────────────────────────────────

print('Vectorization...')

# Allocate the input tensor: one matrix per question, filled with zeros.
# Shape: (1000, 11, 4) → (num_examples, sequence_length, vocab_size)
# Key difference from Parts 1(a) and 1(b): BOTH x and y share the same
# shape — answers are padded to maxlen, not to a shorter DIGITS+1 length.
# dtype=bool saves memory — values are only ever 0 or 1 (one-hot)
x = np.zeros((len(questions), maxlen, len(chars)), dtype=bool)

# Allocate the output tensor — same shape as x, unlike previous parts.
# Shape: (1000, 11, 4)
y = np.zeros((len(questions), maxlen, len(chars)), dtype=bool)

# Fill x: encode each padded question string into its one-hot matrix.
# ctable.encode() returns a (maxlen=2001, 4) matrix; we slot it into row i of x.
print('Vectorizing questions...')
for i, sentence in enumerate(questions):
    x[i] = ctable.encode(sentence, maxlen)

# Fill y: same process for the answers — also encoded to maxlen rows.
print('Vectorising answers...')
for i, sentence in enumerate(expected):
    y[i] = ctable.encode(sentence, maxlen)


## Preparing the Dataset for Training the Model

In [ ]:
# ── Shuffle and Split ─────────────────────────────────────────────────────────

# Shuffle x and y TOGETHER using a shared index array.
# This is the standard safe way to shuffle paired arrays in NumPy —
# if you shuffled x and y separately, each answer would be matched
# to the wrong question, silently corrupting the entire dataset.
indices = np.arange(len(y))    # [0, 1, 2, ..., 49999]
np.random.shuffle(indices)     # e.g., [8312, 441, 27003, ...]
x = x[indices]                 # Reorder x rows using the shuffled index
y = y[indices]                 # Reorder y rows using the SAME shuffled index
                               # → x[i] and y[i] are still a matched pair

# Hold out the last 10% of examples strictly for validation.
# The model NEVER trains on these — they exist only to measure
# how well the model generalises to questions it hasn't seen.
print('Splitting into train and validation sets...')
split_at = len(x) - len(x) // 10
(x_train, x_val) = x[:split_at], x[split_at:]
(y_train, y_val) = y[:split_at], y[split_at:]

# Sanity check: confirm sizes and show a raw one-hot matrix sample
print(f'Size train: {len(x_train)}\tSize val: {len(x_val)}')
print(f'First train input: {x_train[0]}')    # (91, 4) one-hot matrix
print(f'First train answer: {y_train[0]}')   # (91, 4) one-hot matrix


## Assembling the Model

In [ ]:
# ── Model Architecture ────────────────────────────────────────────────────────
# Identical encoder-decoder structure to Parts 1(a) and 1(b).
# The critical dimensional difference: RepeatVector(maxlen) instead of
# RepeatVector(DIGITS+1) — the decoder must output 2001 characters, not 4 or 6.

model = Sequential()

# ENCODER: reads the full 2001-character question, collapses it into a single 128-dim vector.
# input_shape=(2001, 4): 2001 timesteps × 4 chars — the longest sequences of all three parts.
model.add(LSTM(hidden_size, input_shape=(maxlen, len(chars))))
# Output shape after this layer: (batch_size, 128) — the time axis is GONE.

# BRIDGE: copies the single context vector maxlen=2001 times.
# Much larger than Parts 1(a) (4 copies) and 1(b) (6 copies) —
# the decoder must reconstruct a full 2001-character answer string.
model.add(RepeatVector(maxlen))
# Output shape after this layer: (batch_size, 2001, 128)

# DECODER: unrolls the repeated context into an output sequence.
# return_sequences=True returns ALL 2001 hidden states, not just the last one.
model.add(LSTM(hidden_size, return_sequences=True))
# Output shape after this layer: (batch_size, 2001, 128)

# OUTPUT LAYER: applies the same Dense(4) classifier independently to each of
# the 2001 timesteps. softmax converts raw scores into a probability distribution
# over the 4-character vocabulary — the smallest output head of all three parts.
model.add(TimeDistributed(Dense(len(chars), activation='softmax')))
# Output shape after this layer: (batch_size, 2001, 4)

# categorical_crossentropy: standard loss for multi-class classification.
# adam: adaptive learning rate optimiser — robust and fast without manual tuning.
# accuracy: character-level accuracy across all 2001 output positions.
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])
model.summary()


## Training the Model 

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────

# One iteration = one full pass through all 45,000 training examples.
# We manually loop (instead of epochs=50) so we can display live predictions
# after every single epoch.
# Note: no early stopping — runs for a fixed 50 iterations like Part 1(b).
for iteration in range(1, 50):
    print()
    print('-' * 50)
    print('Iteration', iteration)

    # Train for exactly one epoch — epochs=1 is intentional: we need control
    # back after each pass to print predictions manually.
    model.fit(x_train, y_train,
              batch_size=batch_size,    # Process 128 examples per gradient update
              epochs=1,
              validation_data=(x_val, y_val))

    # ── Live Prediction Display ───────────────────────────────────────────────
    # Pick 3 random validation examples and show what the model currently thinks.
    for i in range(3):
        # Sample one random validation example by index
        ind  = np.random.randint(0, len(x_val))
        rowx = x_val[np.array([ind])]    # Shape (1, 2001, 4) — model needs batch dim
        rowy = y_val[np.array([ind])]    # Shape (1, 2001, 4)

        # Run a forward pass — no gradient computation, just inference
        preds = model.predict(rowx, verbose=0)    # Shape (1, 2001, 4) — softmax probs

        # Decode all three from one-hot / probability matrices back to strings.
        # Note: decode() here also prints vocab size (4) on every call —
        # a leftover debug print from development (see CharacterTable above).
        q       = ctable.decode(rowx[0])                     # e.g., '--p---      ...'
        correct = ctable.decode(rowy[0])                     # e.g., 'q-----      ...'
        guess   = ctable.decode(preds[0], calc_argmax=True)  # e.g., 'q-----      ...'

        print('Q', q)        # The input question in pq-system notation
        print('T', correct)  # The ground truth target answer

        # Simpler correctness display than Parts 1(a) and 1(b):
        # full string comparison without stripping — padding is part of the check
        if correct == guess:
            print('☑')
        else:
            print('☒')
        print(guess)         # The model's predicted answer

# Takeaway Messages
* The cannonical example of a `seq2seq` learning task is **language translation**: a seqence represening a sentence in one language is encoded into a latent space (an embedded representation) and then decoded into another language.
* In translation, the **input of characters of variable length** and from a **given alphabet** needs to be converted into an **output also variable in length** and possibly from an altogether **different alphabet**.
* `seq2seq` models generally require **massive amounts** of data to effectively learn their task.